##  Extended Data Figure 2: Comparison of deepUMIcaller and DupCaller across 8 normal bladder urothelium samples

This notebook compares SNVs and INDELs called by **deepUMIcaller** and **DupCaller** at the
sample level. Both callers are run through the same deepCSA pipeline to ensure a fair
comparison.

This notebook is used to generate the following plots:

**a** Venn diagram representing the number of single nucleotide variants (SNVs) identified by either or both pipelines across eight normal bladder urothelium samples.

**b** Breakdown of the 4920 SNVs uniquely identified by DupCaller into those supported by duplex read families with at least one strand formed by only one PCR copy (yellow) and those supported by duplex read families with both strands formed by more than one PCR copy (purple)

**c** Distribution of log-likelihood ratio test values (computed by DupCaller) for SNVs identified by both puiplines (brown) or uniquely identified by DupCaller (orange).

**e** Mutational profiles for common and Only DupCaller SNVs.

**f** Venn diagram representing the number of indels identified by either or both pipelines across eight normal bladder urothelium samples.

In [1]:
# --- 1. Setup: imports -------------------------------------------
from utils import status_counts, save_figure
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from matplotlib.patches import Rectangle
from matplotlib_venn import venn2

In [2]:
# --- Analysis settings --------------------------------------------------------
cohort = "bladder"          # "cord_blood" or "bladder"
mutations = "all_mutation_types"  # "snv", "indel" or "all_mutation_types"

# Base directory for all outputs (plots, tables)
OUTPUT_BASE = Path(
    "../plots"
)

STATUS_COLORS = {
    "Only deepUMI":   "#4E79A7",
    "Only DupCaller": "#F28E2B",
    "Common":         "#A08369",
}

In [3]:

# --- Cohort-specific paths --------------------------------------------------------
if cohort == "bladder":
    samples = [
        "P19_0050_BDO_01", "P19_0050_BTR_01",
        "P19_0051_BDO_01", "P19_0051_BTR_01",
        "P19_0052_BDO_01", "P19_0052_BTR_01",
        "P19_0053_BDO_01", "P19_0053_BTR_01",
    ]
    dupcaller_samples = [ f"{sample}_dupcaller" for sample in samples ]
    comparison_path = f"../data/{cohort}/comparison_deepUMI_dupCaller.mutations.tsv" # Preprocessed with 00_preprocessing.py
    mutations_path = f"../data/{cohort}/all_samples.filtered.somatic.tsv.gz" # Generated with 01_filter_mutation_table.py

## 2. Data loading

Read the three input tables:

| Table | Source | Purpose |
|---|---|---|
| `comparison_df` | DupCaller & deepUMIcaller output| All germline variants called by both callers. Generated with `00_preprocessing.py` |
| `mutations_table` | Combined germline+somatic gzip | Both callers together — needed for trinucleotide context |

In [4]:
# --- Read raw tables --------------------------------------------------------
comparison_df = pd.read_csv(comparison_path, sep="\t", low_memory=False)
mutations_table = pd.read_csv(mutations_path, sep="\t", low_memory=False)

# --- Restrict to specified samples --------------------------------------------------------
mutations_table = mutations_table[(mutations_table["SAMPLE_ID"].isin(samples)) | (mutations_table["SAMPLE_ID"].isin(dupcaller_samples))]

# --- Optional: restrict to SNVs only ----------------------------------------
if mutations == "snv":
    comparison_df = comparison_df[comparison_df["TYPE"] == "SNV"]

elif mutations == "indel":
    comparison_df = comparison_df[comparison_df["TYPE"] == "INDEL"]

print(f"deepUMI variants: {len(comparison_df[(comparison_df["status"] == "Only deepUMI") | (comparison_df["status"] == "Common")]):,}  |  DupCaller variants: {len(comparison_df[(comparison_df["status"] == "Only DupCaller") | (comparison_df["status"] == "Common")]):,}")
print(f"Mutations table rows: {len(mutations_table):,}")

deepUMI variants: 7,100  |  DupCaller variants: 11,292
Mutations table rows: 9,674


## 4. Plotting functions

All visualisation routines. Each function receives pre-computed DataFrames and a
sample name, generates the plot, and saves it to disk.

In [5]:
# --- VENN DIAGRAM plot ----------------------------------------
def plot_venn_pair(
    df: pd.DataFrame,
    sample_name: str,
) -> None:
    """Side-by-side Venn diagrams: all variants vs PASS-in-both variants.

    Left panel shows the full overlap regardless of filter status; right panel
    restricts to variants that pass QC in both callers.
    """
    fig, ax = plt.subplots(1, figsize=(4,4))
    panels = [
        (df, "PASS in both callers"),
    ]

    for ax, (df, title) in zip([ax], panels):
        only_deepumi, only_dupcaller, common = status_counts(df)
        venn = venn2(
            subsets=(only_deepumi, only_dupcaller, common),
            set_labels=("deepUMIcaller", "DupCaller"),
            set_colors=(STATUS_COLORS["Only deepUMI"], STATUS_COLORS["Only DupCaller"]),
            alpha=0.8,
            ax=ax,
        )
        # Colour the overlap region
        if venn.get_patch_by_id("11") is not None:
            venn.get_patch_by_id("11").set_color(STATUS_COLORS["Common"])
            venn.get_patch_by_id("11").set_alpha(0.8)
        ax.set_title(title)

    fig.suptitle(sample_name)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    save_figure(fig, cohort, "plots", mutations, "venn", f"{sample_name.replace(" ", "_")}.pdf", output_base=OUTPUT_BASE)

# --- VIOLIN PLOT COMPARISON ----------------------------------------
def plot_violinplot_pair(
    df: pd.DataFrame,
    sample_name: str,
    column: str,
    caller: str,
) -> None:
    """Violin-plots comparing a numeric feature between Common and caller-unique variants.

    Parameters
    ----------
    column : str
        Column to plot (e.g. "LR_dupcaller", "VAF_deepumi").
    caller : str
        Which caller's unique variants to show ("DupCaller" or "deepUMI").
    """
    fig, ax = plt.subplots(1, figsize=(4, 4), sharey=True)
    panels = [
        (df, "PASS-in-both variants"),
    ]

    for ax, (df, title) in zip([ax], panels):
        common_vals = df[df["status"] == "Common"][column].dropna()
        unique_vals = df[df["status"] == f"Only {caller}"][column].dropna()
        
        data_to_plot = [common_vals if not common_vals.empty else [0], 
                        unique_vals if not unique_vals.empty else [0]]

        vparts = ax.violinplot(
            data_to_plot,
            positions=[1, 2],
            showmeans=False,
            showmedians=True,
            showextrema=False
        )
        
        colors = [STATUS_COLORS["Common"], STATUS_COLORS[f"Only {caller}"]]
        
        # Color the violin bodies
        for pc, color in zip(vparts['bodies'], colors):
            pc.set_facecolor(color)
            pc.set_edgecolor('black')
            pc.set_alpha(0.7)
            
        # Color the lines (median, extrema)
        for partname in ['cmedians']:
            vp = vparts[partname]
            vp.set_edgecolor('black')
            vp.set_linewidth(1)


        # 2. Add Stripplot
        # We use a temporary dataframe for seaborn to handle categorical mapping correctly
        strip_data = pd.DataFrame({
            "val": np.concatenate([common_vals, unique_vals]),
            "grp":  ["Common"] * len(common_vals) + [f"Only {caller}"] * len(unique_vals)
        })
        
        # if not strip_data.empty:
        #     sns.stripplot(
        #         data=strip_data,
        #         x="grp",
        #         y="val",
        #         order=["Extra", "Common", f"Only {caller}"],
        #         jitter=True,
        #         color='black',
        #         alpha=0.4,
        #         ax=ax,
        #         size=3,
        #         zorder=2
        #     )


        ax.set_xticks([1, 2])
        ax.set_xticklabels(["Common", f"Only {caller}"])
        ax.set_ylabel(column)
        ax.set_title(title)
        ax.grid(axis="y", linestyle="--", alpha=0.3)

    fig.suptitle(f"{sample_name} ({column})")
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    save_figure(fig, cohort, "plots", mutations, column, f"violin_{sample_name.replace(" ", "_")}.pdf", output_base=OUTPUT_BASE)

# --- DUPLEX FAMILY HEATMAP ----------------------------------------
def plot_dupcaller_family_heatmap(
    dupcaller_unique_df: pd.DataFrame,
    sample_name: str,
) -> None:
    """Heatmap of F1R2 × F2R1 family-size combinations for DupCaller-unique variants.

    Also shows a stacked bar summarising how many mutations have minimal support
    (family size 2; 1+1) vs higher support (>2; >1+>1).
    """
    df = dupcaller_unique_df.copy()

    # Extract F1R2 and F2R1 counts from the INFO field
    df["F1R2"] = pd.to_numeric(
        df["INFO_dupcaller"].str.extract(r'(?:^|;)F1R2="?([^;"]+)"?')[0],
        errors="coerce",
    )
    df["F2R1"] = pd.to_numeric(
        df["INFO_dupcaller"].str.extract(r'(?:^|;)F2R1="?([^;"]+)"?')[0],
        errors="coerce",
    )

    # Keep only rows with at least 1 read on each strand
    binned_df = (
        df.dropna(subset=["F1R2", "F2R1"])
        .assign(F1R2=lambda x: x["F1R2"].astype(int),
                F2R1=lambda x: x["F2R1"].astype(int))
        .query("F1R2 >= 1 and F2R1 >= 1")
    )

    bin_labels = ["1", "2", "3", "4", "5+"]

    if binned_df.empty:
        heatmap_df = pd.DataFrame(0, index=bin_labels, columns=bin_labels)
        counts = pd.Series({"Using SSC size 1": 0, "Both SSC bigger than 1": 0})
    else:
        # Bin values: 1–4 individually, ≥5 grouped
        binned_df["F1R2_bin"] = np.where(binned_df["F1R2"] >= 5, "5+", binned_df["F1R2"].astype(str))
        binned_df["F2R1_bin"] = np.where(binned_df["F2R1"] >= 5, "5+", binned_df["F2R1"].astype(str))

        combo_counts = (
            binned_df.groupby(["F1R2_bin", "F2R1_bin"])
            .size().reset_index(name="n_mutations")
        )
        heatmap_df = (
            combo_counts.pivot(index="F1R2_bin", columns="F2R1_bin", values="n_mutations")
            .reindex(index=bin_labels, columns=bin_labels, fill_value=0)
            .fillna(0).astype(int)
        )

        # Categorise: minimal support (either strand has only 1 read) vs higher
        binned_df["category"] = np.where(
            (binned_df["F1R2"] == 1) | (binned_df["F2R1"] == 1),
            "Using SSC size 1", "Both SSC bigger than 1",
        )
        counts = binned_df["category"].value_counts().reindex(
            ["Using SSC size 1", "Both SSC bigger than 1"], fill_value=0
        )

    # --- Plot: heatmap + stacked bar -----------------------------------------
    fig, (ax1, ax2) = plt.subplots(
        1, 2, figsize=(7, 5), gridspec_kw={"width_ratios": [3, 1]}
    )

    sns.heatmap(heatmap_df, cmap="viridis", annot=True, fmt="d",
                cbar_kws={"label": "# mutations"}, ax=ax1)
    ax1.set_title(f"DupCaller unique mutation counts\nby each combination of reads\nsupporting each SSC family:\n{sample_name}")
    ax1.set_xlabel("SSC size 1")
    ax1.set_ylabel("SSC size 2")

    # Stacked bar (viridis endpoints for consistency)
    color_low, color_high = "#fde725", "#440154"
    ax2.bar([sample_name], counts["Using SSC size 1"],
            label="Using SSC size 1", color=color_low)
    ax2.bar([sample_name], counts["Both SSC bigger than 1"],
            bottom=counts["Using SSC size 1"],
            label="Both SSC bigger than 1", color=color_high)

    if counts["Using SSC size 1"] > 0:
        ax2.text(0, counts["Using SSC size 1"] / 2,
                 str(int(counts["Using SSC size 1"])),
                 ha="center", va="center", color="black", fontweight="bold")
    if counts["Both SSC bigger than 1"] > 0:
        ax2.text(0, counts["Using SSC size 1"] + counts["Both SSC bigger than 1"] / 2,
                 str(int(counts["Both SSC bigger than 1"])),
                 ha="center", va="center", color="white", fontweight="bold")

    ax2.set_ylabel("Number of mutations\nby amount of support of SSCs")
    ax2.set_title("Single strand\nfamilies' support")
    #ax2.legend()

    plt.tight_layout()
    save_figure(
        fig, cohort, "plots", mutations, "dupcaller_duplex_heatmap",
        f"reads_{sample_name.replace(" ", "_")}.pdf", output_base=OUTPUT_BASE
    )

# --- MUTATION TYPES BARPLOT ----------------------------------------
def plot_mutation_types(
    df: pd.DataFrame,
    sample_name: str,
    mutations: str,
) -> None:
    """Grouped bar chart comparing SNV / INDEL counts by caller status.

    PASS-only variants.
    """
    mutations_list = ["SNV"] if mutations == "snv" else ["SNV", "INDEL"]

    def _prepare_counts(df: pd.DataFrame, panel: str) -> pd.DataFrame:
        x = df.copy()
        x = x[x["TYPE"].isin(mutations_list)]
        out = (x.groupby(["TYPE", "status"], observed=False)
                .size().reset_index(name="n_mutations"))
        out["panel"] = panel
        return out

    plot_df = pd.concat([
        _prepare_counts(df, "PASS only"),
    ], ignore_index=True)

    # Ensure all combinations exist (including zero counts)
    panel_order = ["PASS only"]
    status_order = ["Common", "Only deepUMI", "Only DupCaller"]
    full_idx = pd.MultiIndex.from_product(
        [panel_order, mutations_list, status_order],
        names=["panel", "TYPE", "status"],
    )
    plot_df = (
        plot_df.set_index(["panel", "TYPE", "status"])
        .reindex(full_idx, fill_value=0).reset_index()
    )

    fig, ax = plt.subplots(1, figsize=(10, 4), sharey=True)
    for ax, panel in zip([ax], panel_order):
        d = plot_df[plot_df["panel"] == panel]
        sns.barplot(data=d, x="TYPE", y="n_mutations",
                    hue="status", hue_order=status_order,
                    palette=STATUS_COLORS, ax=ax)
        ax.set_title(panel)
        ax.set_xlabel("Mutation type")
        ax.set_ylabel("Number of mutations")
        ax.grid(axis="y", linestyle="--", alpha=0.4)

        for container in ax.containers:
            labels = [f"{int(v)}" if pd.notna(v) else "" for v in container.datavalues]
            ax.bar_label(container, labels=labels, padding=2, fontsize=9)

    fig.suptitle("Mutation types", y=1.02)
    fig.tight_layout()
    save_figure(
        fig, cohort, "plots", mutations, "mutation_types",
        f"{sample_name.replace(' ', '_')}.pdf", output_base=OUTPUT_BASE
    )

In [6]:
# TODO: CHECK IF WE WANT TO REMOVE THIS ONES -> not used
# --- Boxplot comparing specified column ----------------------------------------
def plot_boxplot_pair(
    merged_all_df: pd.DataFrame,
    merged_pass_df: pd.DataFrame,
    sample_name: str,
    column: str,
    caller: str,
) -> None:
    """Box-plots comparing a numeric feature between Common and caller-unique variants.

    Parameters
    ----------
    column : str
        Column to plot (e.g. "LR_dupcaller", "VAF_deepumi").
    caller : str
        Which caller's unique variants to show ("DupCaller" or "deepUMI").
    """
    fig, axes = plt.subplots(1, 2, figsize=(6, 4))
    panels = [
        (merged_all_df, "All variants"),
        (merged_pass_df, "PASS-in-both variants"),
    ]

    for ax, (df, title) in zip(axes, panels):
        common_vals = df[df["status"] == "Common"][column].dropna()
        unique_vals = df[df["status"] == f"Only {caller}"][column].dropna()

        bp = ax.boxplot(
            [common_vals, unique_vals],
            tick_labels=["Common", f"Only {caller}"],
            patch_artist=True,
            medianprops={"color": "black", "linewidth": 2},
        )
        for patch, color in zip(
            bp["boxes"],
            [STATUS_COLORS["Common"], STATUS_COLORS[f"Only {caller}"]],
        ):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)

        ax.set_ylabel(column)
        ax.set_title(title)
        ax.grid(axis="y", linestyle="--", alpha=0.7)

    fig.suptitle(f"{sample_name}")
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    save_figure(fig, cohort, "plots", mutations, column, f"{sample_name}.png", output_base=OUTPUT_BASE)


## 5. Per-sample analysis

For each sample we:
1. Build the merged variant table (all variants **and** PASS-only).
2. Generate Venn diagrams, box-plots, filter-reason bars, family-size heatmaps,
   and mutation-type bar charts.
3. Export the full merged table as a TSV.

In [7]:
# top_n_deepumi_filters = 10

# for sample in samples:
#     print(f"Processing {sample} …")

#     # --- Subset per sample ---------------------------------------------------
#     sample_deepumi_df = deepumi_df[deepumi_df["SAMPLE_ID"] == sample]
#     sample_deepumi_pass_df = deepumi_clean_df[deepumi_clean_df["SAMPLE_ID"] == sample]
#     sample_dupcaller_df = dupcaller_df[dupcaller_df["SAMPLE_ID"] == f"{sample}_dupcaller"]

#     # --- Merge: all variants & PASS-only -------------------------------------
#     merged_all_df = build_merged_variants(sample_deepumi_df, sample_dupcaller_df)
#     merged_pass_both_df = build_merged_variants(
#         sample_deepumi_df, sample_dupcaller_df,
#         deepumi_pass_only=True, dupcaller_pass_only=True,
#     )

#     # DupCaller-unique variants (PASS filter applied)
#     dupcaller_unique = merged_pass_both_df[
#         merged_pass_both_df["status"] == "Only DupCaller"
#     ].copy()

#     # --- Generate all plots --------------------------------------------------
#     plot_venn_pair(merged_all_df, merged_pass_both_df, sample)
#     plot_boxplot_pair(merged_all_df, merged_pass_both_df, sample, "LR_dupcaller", "DupCaller")
#     plot_violinplot_pair(merged_all_df, merged_pass_both_df, sample, "LR_dupcaller", "DupCaller")
#     plot_boxplot_pair(merged_all_df, merged_pass_both_df, sample, "VAF_deepumi", "deepUMI")
#     plot_deepumi_filter_for_dupcaller_pass(
#         merged_all_df, merged_pass_both_df, sample, top_n=top_n_deepumi_filters,
#     )
#     plot_dupcaller_family_heatmap(dupcaller_unique, sample)
#     plot_mutation_types(merged_all_df, merged_pass_both_df, sample, mutations)

#     # --- Export merged table -------------------------------------------------
#     output_dir = OUTPUT_BASE / cohort / "mutation_comparison" / mutations
#     output_dir.mkdir(parents=True, exist_ok=True)
#     merged_all_df.sort_values(by="status").to_csv(
#         output_dir / f"{sample}.tsv", index=False, sep="\t",
#     )


## 6. Aggregated analysis (all samples pooled)

Same plots as above but merging all samples together to get a cohort-level view.

In [16]:
# Obtain dupcaller unique variants
dupcaller_unique_df = comparison_df[comparison_df["status"] == "Only DupCaller"].copy()

# Generate cohort-level plots
plot_venn_pair(comparison_df, f"All_{cohort}_samples")
plot_mutation_types(comparison_df, "All samples", mutations)
plot_dupcaller_family_heatmap(dupcaller_unique_df, "All samples")
# plot_boxplot_pair(merged_all_df, merged_pass_both_df, "All samples", "LR_dupcaller", "DupCaller")
plot_violinplot_pair(comparison_df, "All samples", "LR_dupcaller", "DupCaller")

## 7. Mutational profiles (SBS-96)

Trinucleotide-context mutational spectra following the standard COSMIC SBS-96
representation. Each bar corresponds to one of the 96 possible single-base
substitutions in their trinucleotide context, grouped by the six pyrimidine
substitution classes (C>A, C>G, C>T, T>A, T>C, T>G).

Profiles are generated separately for:
- **Only deepUMI** – variants exclusive to deepUMIcaller
- **Only DupCaller** – variants exclusive to DupCaller
- **Common** – variants found by both callers

In [6]:
# Standard SBS-96 colour palette
SBS_COLORS = {
    "C>A": "#03bcee",
    "C>G": "#010101",
    "C>T": "#e32926",
    "T>A": "#cac9c9",
    "T>C": "#a1ce63",
    "T>G": "#ebc6c4",
}

# Canonical substitution classes and trinucleotide ordering
SBS_CLASSES = ["C>A", "C>G", "C>T", "T>A", "T>C", "T>G"]
NUCLEOTIDES = ["A", "C", "G", "T"]


def plot_sbs96_profile(
    frequencies: dict[str, float],
    title: str = "Mutational profile",
    yaxis_name: str = "% SBS",
    output_f: str | Path | None = None,
) -> None:
    """Plot an SBS-96 trinucleotide mutational profile.

    Parameters
    ----------
    frequencies : dict
        Mapping from trinucleotide change (e.g. "ACG>T") to its normalised
        frequency.
    title : str
        Plot title (typically includes sample name and mutation count).
    yaxis_name : str
        Label for the y-axis.
    output_f : str or Path, optional
        If given, save the figure to this path at 600 dpi.
    """
    fig, ax = plt.subplots(
        2, sharex="col", figsize=(6, 1.5),
        gridspec_kw={"height_ratios": [0.05, 0.95]},
    )
    plt.subplots_adjust(left=0.1, bottom=0.1, right=0.9, top=0.9,
                        wspace=0.01, hspace=0)
    plt.title(title, fontsize=7, y=1.2)

    # --- Top axis: coloured strips labelling the six substitution classes ------
    ax_top = ax[0]
    for spine in ax_top.spines.values():
        spine.set_visible(False)
    ax_top.set_yticks([])

    fontsize_label = 5
    strip_width = 15.75
    for i, sbs in enumerate(SBS_CLASSES):
        x_offset = i * 16
        ax_top.add_patch(
            Rectangle((x_offset, 0.15 if i == 0 else 0.1), strip_width, 0.5,
                       color=SBS_COLORS[sbs])
        )
        ax_top.text(x_offset + strip_width / 2 - (0 if i == 0 else 1), 0.75,
                    sbs, fontsize=fontsize_label, weight="normal", ha="center")

    ax_bar = ax[1]
    probabilities, colors, labels = [], [], []
    for sbs in SBS_CLASSES:
        ref, alt = sbs.split(">")
        for nuc5 in NUCLEOTIDES:
            for nuc3 in NUCLEOTIDES:
                trinuc = f"{nuc5}{ref}{nuc3}"
                probabilities.append(frequencies.get(f"{trinuc}>{alt}", 0))
                colors.append(SBS_COLORS[sbs])
                labels.append(trinuc)

    ax_bar.bar(range(96), probabilities, width=0.8, color=colors)
    ax_bar.set_xlim(-1, 96)
    ax_bar.set_xticks(range(96))
    ax_bar.set_xticklabels(labels, fontsize=3.5, color="black", va="top", ha="center")
    plt.xticks(rotation=90)
    ax_bar.set_ylabel(yaxis_name, fontsize=6)
    ax_bar.set_yticklabels(ax_bar.get_yticklabels(), fontsize=3.5, color="black",
                           va="center", ha="center")
    for a in ax:
        a.set_axisbelow(True)
        for tick in a.yaxis.get_major_ticks():
            tick.label1.set_fontsize(3.5)
        for spine in a.spines.values():
            spine.set_linewidth(0.25)
        a.tick_params(axis="x", which="both", bottom=False)
        a.tick_params(axis="y", which="both", left=False)
        a.tick_params(axis="y", which="major", pad=2)
        a.tick_params(axis="x", which="major", pad=0)

    if output_f:
        output_f = Path(output_f)
        output_f.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(output_f, bbox_inches="tight", dpi=600)
        plt.close(fig)

In [7]:
# # --- Per-sample SBS-96 profiles (PASS variants only) -------------------------
# # For each sample and each status (Common / Only deepUMI / Only DupCaller),
# # retrieve the trinucleotide context from the combined mutations table and plot

# mutations_table_snvs = mutations_table[mutations_table["TYPE"] == "SNV"].copy()

# for sample in samples:
#     sample_deepumi_df = deepumi_df[deepumi_df["SAMPLE_ID"] == sample]
#     sample_dupcaller_df = dupcaller_df[dupcaller_df["SAMPLE_ID"] == f"{sample}_dupcaller"]

#     merged_pass_both_df = build_merged_variants(
#         sample_deepumi_df, sample_dupcaller_df,
#         deepumi_pass_only=True, dupcaller_pass_only=True,
#     )

#     for status in ["Only deepUMI", "Only DupCaller", "Common"]:
#         subset_ids = merged_pass_both_df[
#             merged_pass_both_df["status"] == status
#         ]["VARIANT_ID"]

#         # Match the correct SAMPLE_ID in the mutations table
#         if status == "Only DupCaller":
#             sample_muts = mutations_table_snvs[
#                 (mutations_table_snvs["SAMPLE_ID"] == f"{sample}_dupcaller")
#                 & mutations_table_snvs["MUT_ID"].isin(subset_ids)
#             ]
#         elif status == "Only deepUMI":
#             sample_muts = mutations_table_snvs[
#                 (mutations_table_snvs["SAMPLE_ID"] == sample)
#                 & mutations_table_snvs["MUT_ID"].isin(subset_ids)
#             ]
#         else:
#             # Common variants appear under both caller SAMPLE_IDs — deduplicate
#             sample_muts = mutations_table_snvs[
#                 mutations_table_snvs["SAMPLE_ID"].isin([sample, f"{sample}_dupcaller"])
#                 & mutations_table_snvs["MUT_ID"].isin(subset_ids)
#             ].drop_duplicates(subset=["MUT_ID"])

#         # Count trinucleotide contexts and normalise
#         context_counts = sample_muts.groupby("CONTEXT_MUT").size().to_frame(sample)
#         total = context_counts[sample].sum()
#         context_counts[sample] = context_counts[sample] / total

#         status_slug = status.lower().replace(" ", "_")
#         plot_sbs96_profile(
#             dict(zip(context_counts.index, context_counts[sample])),
#             title=f"{sample} ({round(total)} muts)",
#             output_f=(
#                 OUTPUT_BASE / cohort / "plots" / "profiles"
#                 / sample.replace("_dupcaller", "") / f"{status_slug}.pdf"
#             ),
#         )

In [12]:
mutations_table[mutations_table['status'] == "Only DupCaller"]

,CHROM,POS,REF,ALT,FILTER,INFO,FORMAT,SAMPLE,DEPTH,ALT_DEPTH,...,FILTER.not_in_exons,FILTER.not_searched_COMPLEX,FILTER.not_searched_SV,FILTER.other_sample_SNP,FILTER.p10,FILTER.pSTD,FILTER.repetitive_variant,SAMPLE_BASE,VARIANT_ID,status
0,chr1,2529741,C,T,not_covered,SAMPLE=P19_0051_BTR_01_dupcaller;,GT:DP:VD:AD:AF:RD:ALD:CDP:CAD:NDP:CDPAM:CADAM:...,"0/1:207:1:206,1:0.00483:103,103:0,1:207:206,1:...",207,1,...,False,False,False,False,False,False,False,P19_0051_BTR_01,chr1:2529741_C>T,Only DupCaller
1,chr1,11059751,C,T,not_covered;not_in_exons,SAMPLE=P19_0052_BDO_01_dupcaller;,GT:DP:VD:AD:AF:RD:ALD:CDP:CAD:NDP:CDPAM:CADAM:...,"0/1:78:1:77,1:0.01282:38,39:0,1:78:77,1:0:78:7...",78,1,...,True,False,False,False,False,False,False,P19_0052_BDO_01,chr1:11059751_C>T,Only DupCaller
2,chr1,11059904,C,T,not_covered,SAMPLE=P19_0052_BDO_01_dupcaller;,GT:DP:VD:AD:AF:RD:ALD:CDP:CAD:NDP:CDPAM:CADAM:...,"0/1:803:1:802,1:0.00125:401,401:0,1:803:802,1:...",803,1,...,False,False,False,False,False,False,False,P19_0052_BDO_01,chr1:11059904_C>T,Only DupCaller
3,chr1,25616838,C,T,not_covered;not_in_exons,SAMPLE=P19_0052_BDO_01_dupcaller;,GT:DP:VD:AD:AF:RD:ALD:CDP:CAD:NDP:CDPAM:CADAM:...,"0/1:58:1:57,1:0.01724:28,29:0,1:58:57,1:0:58:5...",58,1,...,True,False,False,False,False,False,False,P19_0052_BDO_01,chr1:25616838_C>T,Only DupCaller
4,chr1,26729546,T,C,not_in_exons,SAMPLE=P19_0051_BDO_01_dupcaller;,GT:DP:VD:AD:AF:RD:ALD:CDP:CAD:NDP:CDPAM:CADAM:...,"0/1:5173:1:5172,1:0.00019:2586,2586:0,1:5173:5...",5173,1,...,True,False,False,False,False,False,False,P19_0051_BDO_01,chr1:26729546_T>C,Only DupCaller
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9669,chrY,13411337,T,A,not_covered;not_in_exons,SAMPLE=P19_0051_BDO_01_dupcaller;,GT:DP:VD:AD:AF:RD:ALD:CDP:CAD:NDP:CDPAM:CADAM:...,"0/1:298:1:297,1:0.00336:148,149:0,1:298:297,1:...",298,1,...,True,False,False,False,False,False,False,P19_0051_BDO_01,chrY:13411337_T>A,Only DupCaller
9670,chrY,13411404,T,A,not_covered;not_in_exons,SAMPLE=P19_0051_BTR_01_dupcaller;,GT:DP:VD:AD:AF:RD:ALD:CDP:CAD:NDP:CDPAM:CADAM:...,"0/1:262:1:261,1:0.00382:130,131:0,1:262:261,1:...",262,1,...,True,False,False,False,False,False,False,P19_0051_BTR_01,chrY:13411404_T>A,Only DupCaller
9671,chrY,13411515,G,T,not_covered;not_in_exons,SAMPLE=P19_0051_BTR_01_dupcaller;,GT:DP:VD:AD:AF:RD:ALD:CDP:CAD:NDP:CDPAM:CADAM:...,"0/1:73:1:72,1:0.0137:36,36:0,1:73:72,1:0:73:72...",73,1,...,True,False,False,False,False,False,False,P19_0051_BTR_01,chrY:13411515_G>T,Only DupCaller
9672,chrY,13470107,G,C,not_covered;not_in_exons;p10;pSTD,SAMPLE=P19_0053_BDO_01;TYPE=SNV;DP=286;VD=1;AF...,GT:DP:VD:AD:AF:RD:ALD:CDP:CAD:NDP:CDPAM:CADAM:...,"0/1:286:1:285,1:0.0035:268,17:1,0:286:285,1:1:...",286,1,...,True,False,False,False,True,True,False,P19_0053_BDO_01,chrY:13470107_G>C,Only DupCaller


In [13]:
# --- Aggregated SBS-96 profiles (all samples pooled) -------------------------.
sample_label = "All samples"

for status in ["Only DupCaller", "Only deepUMI", "Common"]:
    # Get the (Variant, Sample) pairs for this status — the source of truth
    sample_muts = mutations_table[
        mutations_table["status"] == status
    ][["VARIANT_ID", "SAMPLE_ID", "CONTEXT_MUT"]]

    if sample_muts.empty:
        continue

    context_counts = sample_muts.groupby("CONTEXT_MUT").size().to_frame(sample_label)
    total = int(context_counts[sample_label].sum())
    context_counts[sample_label] = context_counts[sample_label] / total

    status_slug = status.lower().replace(" ", "_")
    plot_sbs96_profile(
        dict(zip(context_counts.index, context_counts[sample_label])),
        title=f"{sample_label} - {status} ({total} muts)",
        output_f=(
            OUTPUT_BASE / cohort / "plots" / "profiles" / "all_samples"
            / f"{status_slug}.pdf"
        ),
    )

/tmp/ipykernel_1684648/522785770.py:78: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax_bar.set_yticklabels(ax_bar.get_yticklabels(), fontsize=3.5, color="black",
/tmp/ipykernel_1684648/522785770.py:78: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax_bar.set_yticklabels(ax_bar.get_yticklabels(), fontsize=3.5, color="black",
/tmp/ipykernel_1684648/522785770.py:78: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax_bar.set_yticklabels(ax_bar.get_yticklabels(), fontsize=3.5, color="black",


## Mutations filtered in deepUMIcaller PASS in DupCaller
There are a total of 4920 PASS mutations unique to DupCaller.
From these, 3142 mutations are probably removed in deepUMIcaller due to the 4;2;2 filter.
For the rest (1778) we need to explore what happens.

Here we will check how many of those are called by deepUMIcaller but filtered during post-processing.

In [42]:
def plot_deepumi_filter_for_dupcaller_pass(
    merged_all_df: pd.DataFrame,
    merged_pass_df: pd.DataFrame,
    sample_name: str,
    top_n: int = 10,
) -> None:
    """Bar chart of deepUMI FILTER reasons for variants that PASS in DupCaller.

    Identifies common variants (present in both callers in the unfiltered merge)
    that are *not* PASS in deepUMI but *are* PASS in DupCaller, then shows the
    most frequent deepUMI filter labels.
    """
    # Variants that already pass in deepUMI (or are deepUMI-unique) — exclude them
    deepumi_pass_ids = merged_pass_df[
        merged_pass_df["status"].isin(["Common", "Only deepUMI"])
    ]["VARIANT_ID"]

    # Common variants in the unfiltered merge that did NOT pass deepUMI
    non_pass_common = merged_all_df[
        (~merged_all_df["VARIANT_ID"].isin(deepumi_pass_ids))
        & (merged_all_df["status"] == "Common")
    ]

    filter_counts = (
        non_pass_common["FILTER_deepumi"]
        .fillna("NA").astype(str)
        .value_counts()
        .sort_values(ascending=False)
        .head(top_n)
    )

    fig, ax = plt.subplots(figsize=(12, 8))

    if filter_counts.empty:
        ax.text(0.5, 0.5, "No Common variants with DupCaller PASS",
                ha="center", va="center", fontsize=11, transform=ax.transAxes)
        ax.axis("off")
    else:
        filter_counts.plot(kind="bar", color=[STATUS_COLORS["Common"]], ax=ax)
        for i, v in enumerate(filter_counts.values):
            ax.text(i, v, str(v), ha="center", va="bottom")
        ax.set_ylabel("Number of variants")
        ax.set_xlabel("deepUMI FILTER")
        ax.set_xticklabels(filter_counts.index, rotation=55, ha="right")
        ax.grid(axis="y", linestyle="--", alpha=0.5)

    ax.set_title(f"Top {top_n} deepUMI filters for DupCaller PASS variants: {sample_name}")
    fig.subplots_adjust(left=0.12, right=0.98, bottom=0.60, top=0.90)
    _save_figure(
        fig, cohort, "plots", mutations, "filters",
        f"deepumi_filter_for_dupcaller_pass_{sample_name}.png",
    )

## deepUMIcaller numbers

There are two things missing to do:
- Calcular la fraccion de mutaciones (SNVs) con el filtro de repetitive regions (across bladder samples)
- Calcular la fraccion de SNVs que hacen overlap con SNP (filtro de gnomAD) (across bladder samples)

In [43]:
def count_mutations(df: pd.DataFrame, filter: str):
    """Count the number of mutations in the deepUMIcaller table with specific filter."""
    has_filter = df[df[f'FILTER.{filter}'] == True]
    no_filter = df[df[f'FILTER.{filter}'] == False]

    return len(has_filter), len(no_filter), len(df)


Low complex repetitive

In [44]:
rows = []
for sample in samples:
    sample_deepumi_df = deepumi_df[deepumi_df["SAMPLE_ID"] == sample]
    has_filter, no_filter, total = count_mutations(sample_deepumi_df, "low_complex_repetitive")
    rows.append({
        "sample": sample,
        "low_complex_repetitive": has_filter,
        "total": total,
        "pct_low_complex_repetitive": round(has_filter / total * 100, 2) if total > 0 else 0.0,
    })

# Add aggregated row for all samples
has_filter_all, no_filter_all, total_all = count_mutations(deepumi_df, "low_complex_repetitive")
rows.append({
    "sample": "All samples",
    "low_complex_repetitive": has_filter_all,
    "total": total_all,
    "pct_low_complex_repetitive": round(has_filter_all / total_all * 100, 2) if total_all > 0 else 0.0,
})

low_complex_df = pd.DataFrame(rows)
low_complex_df

,sample,low_complex_repetitive,total,pct_low_complex_repetitive
0,P19_0050_BDO_01,379,1956,19.38
1,P19_0050_BTR_01,397,2343,16.94
2,P19_0051_BDO_01,305,1605,19.00
3,P19_0051_BTR_01,416,2232,18.64
4,P19_0052_BDO_01,476,1124,42.35
5,P19_0052_BTR_01,393,1052,37.36
6,P19_0053_BDO_01,356,1528,23.30
7,P19_0053_BTR_01,334,1324,25.23
8,All samples,3056,13164,23.21


gnomAD snp

In [45]:
rows = []
for sample in samples:
    sample_deepumi_df = deepumi_df[deepumi_df["SAMPLE_ID"] == sample]
    has_filter, no_filter, total = count_mutations(sample_deepumi_df, "gnomAD_SNP")
    rows.append({
        "sample": sample,
        "gnomAD_SNP": has_filter,
        "total": total,
        "pct_gnomAD_SNP": round(has_filter / total * 100, 2) if total > 0 else 0.0,
    })

# Add aggregated row for all samples
has_filter_all, no_filter_all, total_all = count_mutations(deepumi_df, "gnomAD_SNP")
rows.append({
    "sample": "All samples",
    "gnomAD_SNP": has_filter_all,
    "total": total_all,
    "pct_gnomAD_SNP": round(has_filter_all / total_all * 100, 2) if total_all > 0 else 0.0,
})

low_complex_df = pd.DataFrame(rows)
low_complex_df

,sample,gnomAD_SNP,total,pct_gnomAD_SNP
0,P19_0050_BDO_01,493,1956,25.20
1,P19_0050_BTR_01,484,2343,20.66
2,P19_0051_BDO_01,436,1605,27.17
3,P19_0051_BTR_01,544,2232,24.37
4,P19_0052_BDO_01,716,1124,63.70
5,P19_0052_BTR_01,617,1052,58.65
6,P19_0053_BDO_01,523,1528,34.23
7,P19_0053_BTR_01,503,1324,37.99
8,All samples,4316,13164,32.79
